# Capstone — The Mathematics of a Production Multimodal Financial RAG System

The retrieval stack arrived as a dozen separate topics — lexical BM25, dense MIPS, IVF/PQ/HNSW,
late interaction served by PLAID, reciprocal-rank fusion. This capstone **composes** the published
stack into one finance pipeline and proves the three laws that govern it end to end, with exactly
**one exact statement** (a collapse anchor) and **three measured laws**.

The notebook imports the reference module `capstone_multimodal_financial_rag.py`, which owns every
number; the cells below walk Movement 0 (the shared corpus + legs) through Movement 3 (budget
allocation), running each pedagogical claim as an assertion.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks" / "capstone-multimodal-financial-rag"))
import capstone_multimodal_financial_rag as cap

# The cascade composition law, in one line: end-to-end retention is the product of per-stage retentions.
print("cascade_recall([0.8, 0.5, 0.9]) =", cap.cascade_recall([0.8, 0.5, 0.9]))
print("over_fetch_factor               =", round(cap.over_fetch_factor([0.8, 0.5, 0.9]), 3))

## Movement 0 — one shared multi-view corpus, three heterogeneous legs

Every document is one bag of unit-norm tokens (`token_corpus`, the PLAID cloud), seen three ways:
the **lexical** leg matches a partial window of the tokens' centroid ids (BM25 over a quantized
vocabulary), the **dense** leg pools a disjoint window into a single vector, and the
**late-interaction** leg scores a third window by cheap centroid-MaxSim. The exact end-scorer is
brute-force MaxSim, so the ground truth is **neutral** — no single candidate-generation leg is the
oracle, which is what lets the legs genuinely complement.

In [ ]:
corpus = cap._corpus()
per_leg = {name: round(cap.mean_leg_recall(corpus, fn), 3) for name, fn in cap.LEGS.items()}
print("per-leg recall@10 vs the MaxSim truth:", per_leg)
cap.test_legs_share_one_corpus_and_truth()
print("legs rank one shared corpus against one shared truth: OK")

## Movement 1 — cascade recall composition

A served pipeline is a sequence of lossy stages; end-to-end retention is the **product** of the
per-stage retentions (exact under independent survival), and the front-end over-fetch is the
algebraic reciprocal `1/∏rᵢ` of one negative-binomial `k/r` law on the composite retention. Under
the **positive dependence** real queries exhibit, the independent product is a conservative **lower
bound** (FKG/Harris) — verified here by a bivariate-normal copula across both signs of correlation,
with equality at independence. The collapse anchor: every stage retaining all → recall 1.0.

In [ ]:
cap.test_cascade_recall_is_product()
cap.test_overfetch_is_reciprocal_identity()
cap.test_independence_recovers_product()
cap.test_dependence_direction()
cap.test_measured_stages_compose_as_lower_bound()
cap.test_cascade_collapse_anchor()

stages = cap.measure_cascade_stages(corpus, nprobe=8, keep=40)
print("measured stage retentions r1,r2,r3 =",
      tuple(round(stages[k], 3) for k in ("r1", "r2", "r3")))
print("product (lower bound) =", round(stages["product"], 3),
      " <=  measured end-to-end recall =", round(stages["measured_end"], 3))
print("FKG dependence sweep (rho, R_true - R_indep):")
for r in cap.dependence_sweep(cap.DEMO_R1, cap.DEMO_R2, (-0.6, 0.0, 0.6, 0.9)):
    print("   rho=%+.1f  gap=%+.3f" % (r["rho"], r["gap"]))

## Movement 2 — hybrid fusion gain

Reciprocal-rank fusion of the heterogeneous legs can **beat the best single leg**, and the gain
grows as the legs **de-correlate** (each recalls true neighbors the others miss). But it is a
*condition*, not a theorem: a false positive **co-endorsed by both legs** can displace a true
neighbor — under RRF's `c=60` a lone top vote (`1/61`) is weaker than two mid votes (`2/63`), so the
fused ranking flips below the strong leg. We build and run that flip before trusting the gain.

In [ ]:
cap.test_fusion_gain_under_complementarity()
cap.test_dominated_leg_flip()
cap.test_decorrelation_endpoints()
cap.test_combsum_scale_sensitivity()

fg = cap.fusion_gain(corpus)
print("per-leg:", {k: round(v, 3) for k, v in fg["per_leg"].items()})
print("fused = %.3f  >  best leg = %.3f   (gain %+.3f)" % (fg["fused"], fg["best_leg"], fg["gain"]))
flip = cap.dominated_leg_demo()
print("dominated-leg flip: fused top-3 =", flip["fused_top3"],
      " rho_strong=%.2f -> rho_fused=%.3f (flips=%s)" % (flip["rho_strong"], flip["rho_fused"], flip["flips"]))
print("no co-endorsement => no flip, recall =", round(flip["rho_fused_no_coendorse"], 3))

## Movement 3 — end-to-end budget allocation (water-filling)

Each leg has a concave retention curve costed in a common unit (distance-computations/query), the
late-interaction leg being far more expensive per scored document. Maximizing the separable model
`R = ∏ gᵢ(cᵢ)` under a budget equalizes the **marginal log-recall per cost** across active stages —
water-filling. It beats naive uniform allocation and crushes all-in-one (a starved factor tanks the
product). The separable product is, by the Movement-1 caveat, a lower bound the real fused pipeline
sits above. The exact anchor: full budget everywhere → every gᵢ → 1 → recall 1.0.

In [ ]:
cap.test_logconcave_on_grid()
cap.test_waterfilling_optimality_and_beats_baselines()
cap.test_equal_marginal_also_from_unlogged_product()

curves = cap.stage_gain_curves(corpus)
B = cap.demo_budget(curves)
wf, un = cap.allocate_budget(curves, B), cap.uniform_alloc(curves, B)
aio = max(cap.end_to_end_recall(cap.all_in_one_alloc(curves, B, s), curves) for s in curves)
print("budget B = %.0f comps/query" % B)
print("water-filling R = %.3f  |  uniform R = %.3f  |  best all-in-one R = %.3f"
      % (cap.end_to_end_recall(wf, curves), cap.end_to_end_recall(un, curves), aio))
print("marginal log-recall LEVELS at the optimum:",
      {s: round(m, 5) for s, m in cap.marginal_log_recall(wf, curves).items()})

## Storage, the collapse anchor, and the baked viz constants

A representative ColBERT-scale account: raw multi-vector storage is **32×** a single-vector index;
PLAID's centroid-id-plus-residual collapses it back to **~1×**. Finally, `viz_constants()` prints
every number `CapstoneLaboratory.tsx` mirrors to the decimal.

In [ ]:
cap.test_storage_collapse()
print("storage:", cap.storage_accounting())
print("collapse anchor recall =", round(cap.collapse_anchor(corpus)["recall"], 3))
print()
cap.viz_constants()